In [1]:
import os
import json
import re
import pandas as pd

# object oriented approach to working with paths
from pathlib import Path

import duckdb

In [2]:
# ensuring that os.chdir is idempotent and that we are in the project root directory,
# not inside the notebooks directory
if 'notebooks' not in os.listdir(Path.cwd()):
    print("Still inside notebooks directory, changing to project root directory.")
    os.chdir(Path.cwd().parent)
    print("Current working directory after change: ", Path.cwd())
else:
    print(f"Already in parent directory (current working directory: {Path.cwd()})")


# load config
from src.io.load_config import load_config
sw_config = load_config()['space_weather']
omni_config = load_config()['omni']

Still inside notebooks directory, changing to project root directory.
Current working directory after change:  d:\data-sci-projects\space-weather-project-scrub


# Import audit-table preprocessing functions

In [3]:
from src.preprocess.omni_preproc import\
    _discover_successful_manifests,\
    _read_processed_run_ids,\
    pick_oldest_unprocessed_successful_run,\
    _discover_chunk_paths

# Prep: Ensure that no long directory has been created yet

In [34]:
raw_dataset_dir = "temp/omni-preproc-smoke/raw/"
audit_output_dir = "temp/omni-preproc-smoke/preproc/long-observations"
_discover_successful_manifests(raw_dataset_dir)

[WindowsPath('temp/omni-preproc-smoke/raw/run_id=20260803T025211Z/_manifest.json'),
 WindowsPath('temp/omni-preproc-smoke/raw/run_id=20260803T025255Z/_manifest.json'),
 WindowsPath('temp/omni-preproc-smoke/raw/run_id=20260806T052236Z/_manifest.json')]

In [ ]:
if os.listdir(audit_output_dir):
    raise Exception("Please delete audit table before this smoke test")

Exception: Please delete long table before this smoke test

# Phase A: Test run & manifest selection helpers & no-output edge cases

```
# manifest and selection helpers
_discover_successful_manifests(...)
_discover_chunk_paths(...)

# edge cases for non-existent long table
_read_processed_run_ids(...)
pick_oldest_unprocessed_successful_run(...)

```

A missing or empty long-audit directory means that no successful runs have been processed.

In [5]:
_read_processed_run_ids(audit_output_dir) == set()

True

Because the processed set is empty, `pick_oldest_unprocessed_successful_run` should return the oldest successful raw run.

In [6]:
_discover_successful_manifests(raw_dataset_dir)

[WindowsPath('temp/omni-preproc-smoke/raw/run_id=20260803T025211Z/_manifest.json'),
 WindowsPath('temp/omni-preproc-smoke/raw/run_id=20260803T025255Z/_manifest.json'),
 WindowsPath('temp/omni-preproc-smoke/raw/run_id=20260806T052236Z/_manifest.json')]

In [7]:
first_unprocessed_run = pick_oldest_unprocessed_successful_run(
    raw_dataset_dir,
    audit_output_dir,
)

# first occurence of the manifest path associated with the 1st unprocessed run
first_unprocessed_run_manifest_path = next(
    (x for x in _discover_successful_manifests(raw_dataset_dir) if first_unprocessed_run in x.as_posix()),
    None) 

print(f'First unprocessed run: {first_unprocessed_run}\nAnd its path: {first_unprocessed_run_manifest_path}')

First unprocessed run: 20260803T025211Z
And its path: temp\omni-preproc-smoke\raw\run_id=20260803T025211Z\_manifest.json


# Phase B: Query-Only Validation
Build the long-observation query for the selected run and execute it directly through DuckDB without COPY.

Inspect:
- exact schema;
- output row count;
- distinct run IDs;
- distinct parameter names;
- source-fill count;
- sentinel count;
- a small ordered sample.

Confirm that the selected run produces the expected relation before testing filesystem materialization.

In [8]:
from src.preprocess.omni_preproc import build_long_observation_select_sql, _read_manifest_json

In [9]:
print(f"Printing long table from only processing run_id={first_unprocessed_run}")
con = duckdb.connect()
first_long_table_test = con.execute(
    build_long_observation_select_sql(
        manifest_paths=[first_unprocessed_run_manifest_path],
        chunk_paths=_discover_chunk_paths(first_unprocessed_run_manifest_path)

    )
).fetch_df()
first_long_table_test

Printing long table from only processing run_id=20260803T025211Z


,dataset_id,run_id,chunk_file,source_row_number,observation_time_utc,parameter_name,raw_value,source_fill_value,units,parameter_type,is_source_fill
0,OMNI_HRO2_1MIN,20260803T025211Z,chunk_20260707T000000Z__20260708T004400Z.json,1,2026-07-07 00:00:00,IMF,52.0,99.0,None,integer,False
1,OMNI_HRO2_1MIN,20260803T025211Z,chunk_20260707T000000Z__20260708T004400Z.json,1,2026-07-07 00:00:00,PLS,52.0,99.0,None,integer,False
2,OMNI_HRO2_1MIN,20260803T025211Z,chunk_20260707T000000Z__20260708T004400Z.json,1,2026-07-07 00:00:00,IMF_PTS,1.0,999.0,None,integer,False
3,OMNI_HRO2_1MIN,20260803T025211Z,chunk_20260707T000000Z__20260708T004400Z.json,1,2026-07-07 00:00:00,PLS_PTS,1.0,999.0,None,integer,False
4,OMNI_HRO2_1MIN,20260803T025211Z,chunk_20260707T000000Z__20260708T004400Z.json,1,2026-07-07 00:00:00,percent_interp,100.0,999.0,None,integer,False
...,...,...,...,...,...,...,...,...,...,...,...
62323,OMNI_HRO2_1MIN,20260803T025211Z,chunk_20260707T000000Z__20260708T004400Z.json,1484,2026-07-08 00:43:00,AU_INDEX,99999.0,99999.0,nT,integer,True
62324,OMNI_HRO2_1MIN,20260803T025211Z,chunk_20260707T000000Z__20260708T004400Z.json,1484,2026-07-08 00:43:00,SYM_D,99999.0,99999.0,nT,integer,True
62325,OMNI_HRO2_1MIN,20260803T025211Z,chunk_20260707T000000Z__20260708T004400Z.json,1484,2026-07-08 00:43:00,SYM_H,99999.0,99999.0,nT,integer,True
62326,OMNI_HRO2_1MIN,20260803T025211Z,chunk_20260707T000000Z__20260708T004400Z.json,1484,2026-07-08 00:43:00,ASY_D,99999.0,99999.0,nT,integer,True


In [10]:
display(first_long_table_test.run_id.unique())

array(['20260803T025211Z'], dtype=object)

In [11]:
_read_manifest_json(first_unprocessed_run_manifest_path)

{'artifacts': {'chunks': [{'chunk_end_utc_str': '2026-07-08T00:44:00Z',
    'chunk_start_utc_str': '2026-07-07T00:00:00Z',
    'file': 'chunk_20260707T000000Z__20260708T004400Z.json',
    'hapi_status_code': 1200,
    'hapi_status_message': 'OK',
    'rows': 1484}],
  'info_file': 'hapi_info.json'},
 'error': None,
 'ingestion': {'chunk_days': 10, 'sleep_s': 5, 'timeout_s': 120},
 'preflight_warnings': ['Requested date interval was clipped to the available dataset interval.'],
 'request': {'effective_end_utc': '2026-07-08T00:44:00Z',
  'effective_start_utc': '2026-07-07T00:00:00Z',
  'parameters': ['Time',
   'IMF',
   'PLS',
   'IMF_PTS',
   'PLS_PTS',
   'percent_interp',
   'Timeshift',
   'RMS_Timeshift',
   'RMS_phase',
   'Time_btwn_obs',
   'F',
   'BX_GSE',
   'BY_GSE',
   'BZ_GSE',
   'BY_GSM',
   'BZ_GSM',
   'RMS_SD_B',
   'RMS_SD_fld_vec',
   'flow_speed',
   'Vx',
   'Vy',
   'Vz',
   'proton_density',
   'T',
   'NaNp_Ratio',
   'Pressure',
   'E',
   'Beta',
   'Mach_num

# Phase C: One Incremental Write

In [12]:
from src.preprocess.omni_preproc import increment_successful_run

In [13]:
first_increment = increment_successful_run(
    raw_dataset_dir,
    audit_output_dir,
)

In [14]:
first_long_table = con.execute(f"""
    SELECT * 
    FROM read_parquet('{first_increment.as_posix()}/**/*.parquet')
""").fetch_df()
first_long_table

,dataset_id,chunk_file,source_row_number,observation_time_utc,parameter_name,raw_value,source_fill_value,units,parameter_type,is_source_fill,run_id
0,OMNI_HRO2_1MIN,chunk_20260707T000000Z__20260708T004400Z.json,1,2026-07-07 00:00:00,IMF,52.0,99.0,None,integer,False,20260803T025211Z
1,OMNI_HRO2_1MIN,chunk_20260707T000000Z__20260708T004400Z.json,1,2026-07-07 00:00:00,PLS,52.0,99.0,None,integer,False,20260803T025211Z
2,OMNI_HRO2_1MIN,chunk_20260707T000000Z__20260708T004400Z.json,1,2026-07-07 00:00:00,IMF_PTS,1.0,999.0,None,integer,False,20260803T025211Z
3,OMNI_HRO2_1MIN,chunk_20260707T000000Z__20260708T004400Z.json,1,2026-07-07 00:00:00,PLS_PTS,1.0,999.0,None,integer,False,20260803T025211Z
4,OMNI_HRO2_1MIN,chunk_20260707T000000Z__20260708T004400Z.json,1,2026-07-07 00:00:00,percent_interp,100.0,999.0,None,integer,False,20260803T025211Z
...,...,...,...,...,...,...,...,...,...,...,...
62323,OMNI_HRO2_1MIN,chunk_20260707T000000Z__20260708T004400Z.json,1484,2026-07-08 00:43:00,AU_INDEX,99999.0,99999.0,nT,integer,True,20260803T025211Z
62324,OMNI_HRO2_1MIN,chunk_20260707T000000Z__20260708T004400Z.json,1484,2026-07-08 00:43:00,SYM_D,99999.0,99999.0,nT,integer,True,20260803T025211Z
62325,OMNI_HRO2_1MIN,chunk_20260707T000000Z__20260708T004400Z.json,1484,2026-07-08 00:43:00,SYM_H,99999.0,99999.0,nT,integer,True,20260803T025211Z
62326,OMNI_HRO2_1MIN,chunk_20260707T000000Z__20260708T004400Z.json,1484,2026-07-08 00:43:00,ASY_D,99999.0,99999.0,nT,integer,True,20260803T025211Z


# Phase D: Processed-State Transition

Long audit table should have existed (first run is processed)

In [15]:
# first verify that the long table already has data from one run
_read_processed_run_ids(audit_output_dir)

{'20260803T025211Z'}

# Phase E: Second Increment

The oldest run picker algorithm should have picked the **second** oldest successful run not yet being processed.

In [16]:
second_unprocessed_run = pick_oldest_unprocessed_successful_run(
    raw_dataset_dir,
    audit_output_dir,
)

# first occurence of the manifest path associated with the 1st unprocessed run
second_unprocessed_run_manifest_path = next(
    (x for x in _discover_successful_manifests(raw_dataset_dir) if second_unprocessed_run in x.as_posix()),
    None) 

print(f'Second unprocessed run: {second_unprocessed_run}\nAnd its path: {second_unprocessed_run_manifest_path}')

Second unprocessed run: 20260803T025255Z
And its path: temp\omni-preproc-smoke\raw\run_id=20260803T025255Z\_manifest.json


In [17]:
con.execute(
    build_long_observation_select_sql(
        manifest_paths=[second_unprocessed_run_manifest_path],
        chunk_paths=_discover_chunk_paths(second_unprocessed_run_manifest_path)

    )
).fetch_df()

,dataset_id,run_id,chunk_file,source_row_number,observation_time_utc,parameter_name,raw_value,source_fill_value,units,parameter_type,is_source_fill
0,OMNI_HRO2_1MIN,20260803T025255Z,chunk_20260706T000000Z__20260707T000000Z.json,1,2026-07-06 00:00:00,IMF,52.0,99.0,None,integer,False
1,OMNI_HRO2_1MIN,20260803T025255Z,chunk_20260706T000000Z__20260707T000000Z.json,1,2026-07-06 00:00:00,PLS,99.0,99.0,None,integer,True
2,OMNI_HRO2_1MIN,20260803T025255Z,chunk_20260706T000000Z__20260707T000000Z.json,1,2026-07-06 00:00:00,IMF_PTS,1.0,999.0,None,integer,False
3,OMNI_HRO2_1MIN,20260803T025255Z,chunk_20260706T000000Z__20260707T000000Z.json,1,2026-07-06 00:00:00,PLS_PTS,999.0,999.0,None,integer,True
4,OMNI_HRO2_1MIN,20260803T025255Z,chunk_20260706T000000Z__20260707T000000Z.json,1,2026-07-06 00:00:00,percent_interp,100.0,999.0,None,integer,False
...,...,...,...,...,...,...,...,...,...,...,...
60475,OMNI_HRO2_1MIN,20260803T025255Z,chunk_20260706T000000Z__20260707T000000Z.json,1440,2026-07-06 23:59:00,AU_INDEX,99999.0,99999.0,nT,integer,True
60476,OMNI_HRO2_1MIN,20260803T025255Z,chunk_20260706T000000Z__20260707T000000Z.json,1440,2026-07-06 23:59:00,SYM_D,99999.0,99999.0,nT,integer,True
60477,OMNI_HRO2_1MIN,20260803T025255Z,chunk_20260706T000000Z__20260707T000000Z.json,1440,2026-07-06 23:59:00,SYM_H,99999.0,99999.0,nT,integer,True
60478,OMNI_HRO2_1MIN,20260803T025255Z,chunk_20260706T000000Z__20260707T000000Z.json,1440,2026-07-06 23:59:00,ASY_D,99999.0,99999.0,nT,integer,True


Run `increment_successful_run()` again.

In [18]:
second_increment = increment_successful_run(
    raw_dataset_dir,
    audit_output_dir,
)
second_increment

WindowsPath('temp/omni-preproc-smoke/preproc/long-observations')

In [19]:
second_long_table = con.execute(f"""
    SELECT * 
    FROM read_parquet('{first_increment.as_posix()}/**/*.parquet')
""").fetch_df()
second_long_table

,dataset_id,chunk_file,source_row_number,observation_time_utc,parameter_name,raw_value,source_fill_value,units,parameter_type,is_source_fill,run_id
0,OMNI_HRO2_1MIN,chunk_20260707T000000Z__20260708T004400Z.json,1,2026-07-07 00:00:00,IMF,52.0,99.0,None,integer,False,20260803T025211Z
1,OMNI_HRO2_1MIN,chunk_20260707T000000Z__20260708T004400Z.json,1,2026-07-07 00:00:00,PLS,52.0,99.0,None,integer,False,20260803T025211Z
2,OMNI_HRO2_1MIN,chunk_20260707T000000Z__20260708T004400Z.json,1,2026-07-07 00:00:00,IMF_PTS,1.0,999.0,None,integer,False,20260803T025211Z
3,OMNI_HRO2_1MIN,chunk_20260707T000000Z__20260708T004400Z.json,1,2026-07-07 00:00:00,PLS_PTS,1.0,999.0,None,integer,False,20260803T025211Z
4,OMNI_HRO2_1MIN,chunk_20260707T000000Z__20260708T004400Z.json,1,2026-07-07 00:00:00,percent_interp,100.0,999.0,None,integer,False,20260803T025211Z
...,...,...,...,...,...,...,...,...,...,...,...
122803,OMNI_HRO2_1MIN,chunk_20260706T000000Z__20260707T000000Z.json,1440,2026-07-06 23:59:00,AU_INDEX,99999.0,99999.0,nT,integer,True,20260803T025255Z
122804,OMNI_HRO2_1MIN,chunk_20260706T000000Z__20260707T000000Z.json,1440,2026-07-06 23:59:00,SYM_D,99999.0,99999.0,nT,integer,True,20260803T025255Z
122805,OMNI_HRO2_1MIN,chunk_20260706T000000Z__20260707T000000Z.json,1440,2026-07-06 23:59:00,SYM_H,99999.0,99999.0,nT,integer,True,20260803T025255Z
122806,OMNI_HRO2_1MIN,chunk_20260706T000000Z__20260707T000000Z.json,1440,2026-07-06 23:59:00,ASY_D,99999.0,99999.0,nT,integer,True,20260803T025255Z


In [20]:
second_long_table.run_id.unique()

array(['20260803T025211Z', '20260803T025255Z'], dtype=object)

In [25]:
os.listdir('temp/omni-preproc-smoke/preproc/long-observations/run_id=20260803T025211Z')

['data_0.parquet']

# Rebuild smoke test

In [31]:
from src.preprocess.omni_preproc import rebuild_successful_runs

In [29]:
# assert not os.listdir(audit_output_dir)

In [30]:
_read_processed_run_ids(audit_output_dir)

{'20260803T025211Z', '20260803T025255Z'}

In [32]:
first_rebuild = rebuild_successful_runs(
    raw_dataset_dir,
    audit_output_dir,
)

In [36]:
audit_output_dir

'temp/omni-preproc-smoke/preproc/long-observations'

In [33]:
_read_processed_run_ids(audit_output_dir)

{'20260803T025211Z', '20260803T025255Z', '20260806T052236Z'}

In [43]:
rebuild_smoke_test = con.execute(f"""
    SELECT * 
    FROM read_parquet('{audit_output_dir}/**/*.parquet')
""").fetch_df()
rebuild_smoke_test

,dataset_id,chunk_file,source_row_number,observation_time_utc,parameter_name,raw_value,source_fill_value,units,parameter_type,is_source_fill,run_id
0,OMNI_HRO2_1MIN,chunk_20260707T000000Z__20260708T004400Z.json,1,2026-07-07 00:00:00,IMF,52.0,99.0,None,integer,False,20260803T025211Z
1,OMNI_HRO2_1MIN,chunk_20260707T000000Z__20260708T004400Z.json,1,2026-07-07 00:00:00,PLS,52.0,99.0,None,integer,False,20260803T025211Z
2,OMNI_HRO2_1MIN,chunk_20260707T000000Z__20260708T004400Z.json,1,2026-07-07 00:00:00,IMF_PTS,1.0,999.0,None,integer,False,20260803T025211Z
3,OMNI_HRO2_1MIN,chunk_20260707T000000Z__20260708T004400Z.json,1,2026-07-07 00:00:00,PLS_PTS,1.0,999.0,None,integer,False,20260803T025211Z
4,OMNI_HRO2_1MIN,chunk_20260707T000000Z__20260708T004400Z.json,1,2026-07-07 00:00:00,percent_interp,100.0,999.0,None,integer,False,20260803T025211Z
...,...,...,...,...,...,...,...,...,...,...,...
122804,OMNI_HRO2_1MIN,chunk_20260706T000000Z__20260707T000000Z.json,1440,2026-07-06 23:59:00,SYM_D,99999.0,99999.0,nT,integer,True,20260803T025255Z
122805,OMNI_HRO2_1MIN,chunk_20260706T000000Z__20260707T000000Z.json,1440,2026-07-06 23:59:00,SYM_H,99999.0,99999.0,nT,integer,True,20260803T025255Z
122806,OMNI_HRO2_1MIN,chunk_20260706T000000Z__20260707T000000Z.json,1440,2026-07-06 23:59:00,ASY_D,99999.0,99999.0,nT,integer,True,20260803T025255Z
122807,OMNI_HRO2_1MIN,chunk_20260706T000000Z__20260707T000000Z.json,1440,2026-07-06 23:59:00,ASY_H,99999.0,99999.0,nT,integer,True,20260803T025255Z


In [44]:
rebuild_smoke_test.groupby('run_id')['source_row_number'].count()

run_id
20260803T025211Z    62328
20260803T025255Z    60480
20260806T052236Z        0
Name: source_row_number, dtype: Int64

In [ ]:
for detected_run_id in rebuild_smoke_test.run_id.unique():

    # note we only unnest non-time parameters so the total exploded data
    # rows is multiplied by one less than the parameter list length
    raw_data_row_counts = con.execute(f"""
        SELECT UNNEST(data)
        FROM read_json('{raw_dataset_dir}/run_id={detected_run_id}/chunk*.json')
    """).fetch_df().shape[0]

    parameter_list_counts = con.execute(f"""
        SELECT UNNEST(parameters)
        FROM read_json('{raw_dataset_dir}/run_id={detected_run_id}/chunk*.json')
    """).fetch_df().shape[0]

    print(f"---RUN ID: {detected_run_id}---")
    expected_audit_table_counts = raw_data_row_counts * (parameter_list_counts - 1)
    print(f"---RAW JSON DATA COUNTS: {raw_data_row_counts}---")
    print(f"---PARAMETER LIST COUNTS (INCL. TIME): {parameter_list_counts}---")
    print(f"---EXPECTED # ROWS IN AUDIT TABLE: {expected_audit_table_counts}---\n")    

---RUN ID: 20260803T025211Z---
---RAW JSON DATA COUNTS: 1484---
---PARAMETER LIST COUNTS (INCL. TIME): 43---
---EXPECTED # ROWS IN AUDIT TABLE: 62328---

---RUN ID: 20260803T025255Z---
---RAW JSON DATA COUNTS: 1440---
---PARAMETER LIST COUNTS (INCL. TIME): 43---
---EXPECTED # ROWS IN AUDIT TABLE: 60480---

---RUN ID: 20260806T052236Z---
---RAW JSON DATA COUNTS: 1440---
---PARAMETER LIST COUNTS (INCL. TIME): 1---
---EXPECTED # ROWS IN AUDIT TABLE: 0---



This makes sense: Last run only requested for `Time` parameter and since we only unnest non-time parameters -> sentinel row.

Also SQL tip: `parameters[2:]` evaluates to an empty list `[]` (DuckDB uses 1-based indexing and sine), so the `CROSS JOIN UNNEST` thing in 
```
def _build_parameter_definitions_select_sql() -> str:
    """Build positional metadata for each non-time parameter."""
    return """
        SELECT
            dataset_id,
            run_id,
            filename,
            parameter_index,
            parameter.name AS parameter_name,
            CAST(parameter.fill AS DOUBLE) AS source_fill_value,
            parameter.units AS units,
            parameter.type AS parameter_type
        FROM successful_chunks
        CROSS JOIN UNNEST(parameters[2:])
            WITH ORDINALITY AS definitions(
                parameter,
                parameter_index
            )
    """.strip()
```

will produce **no output rows**.


# Rebuild smoke test idempotency

In [ ]:
second_rebuild = rebuild_successful_runs(
    raw_dataset_dir,
    audit_output_dir,
)

In [76]:
second_rebuild.as_posix() == audit_output_dir

True

In [77]:
con.execute(f"""
    SELECT * 
    FROM read_parquet('{second_rebuild}/**/*.parquet')
""").fetch_df().equals(rebuild_smoke_test)

True

# Edge case: what if we forgot to request for `Time` as a parameter?

In [82]:
third_rebuild_accounting_for_edge_case = rebuild_successful_runs(
    raw_dataset_dir,
    audit_output_dir,
)

In [85]:
third_rebuild_smoke_test = con.execute(f"""
    SELECT * 
    FROM read_parquet('{third_rebuild_accounting_for_edge_case}/**/*.parquet')
""").fetch_df()

In [88]:
for detected_run_id in third_rebuild_smoke_test.run_id.unique():

    # note we only unnest non-time parameters so the total exploded data
    # rows is multiplied by one less than the parameter list length
    raw_data_row_counts = con.execute(f"""
        SELECT UNNEST(data)
        FROM read_json('{raw_dataset_dir}/run_id={detected_run_id}/chunk*.json')
    """).fetch_df().shape[0]

    parameter_list_counts = con.execute(f"""
        SELECT UNNEST(parameters)
        FROM read_json('{raw_dataset_dir}/run_id={detected_run_id}/chunk*.json')
    """).fetch_df().shape[0]

    print(f"---RUN ID: {detected_run_id}---")
    expected_audit_table_counts = raw_data_row_counts * (parameter_list_counts - 1)
    print(f"---RAW JSON DATA COUNTS: {raw_data_row_counts}---")
    print(f"---PARAMETER LIST COUNTS (INCL. TIME): {parameter_list_counts}---")
    print(f"---EXPECTED # ROWS IN AUDIT TABLE: {expected_audit_table_counts}---\n")    

---RUN ID: 20260803T025211Z---
---RAW JSON DATA COUNTS: 1484---
---PARAMETER LIST COUNTS (INCL. TIME): 43---
---EXPECTED # ROWS IN AUDIT TABLE: 62328---

---RUN ID: 20260803T025255Z---
---RAW JSON DATA COUNTS: 1440---
---PARAMETER LIST COUNTS (INCL. TIME): 43---
---EXPECTED # ROWS IN AUDIT TABLE: 60480---

---RUN ID: 20260806T052236Z---
---RAW JSON DATA COUNTS: 1440---
---PARAMETER LIST COUNTS (INCL. TIME): 1---
---EXPECTED # ROWS IN AUDIT TABLE: 0---

---RUN ID: 20260811T234025Z---
---RAW JSON DATA COUNTS: 1440---
---PARAMETER LIST COUNTS (INCL. TIME): 2---
---EXPECTED # ROWS IN AUDIT TABLE: 1440---



In [87]:
third_rebuild_smoke_test.groupby('run_id')['source_row_number'].count()

run_id
20260803T025211Z    62328
20260803T025255Z    60480
20260806T052236Z        0
20260811T234025Z     1440
Name: source_row_number, dtype: Int64